# 🔬 Chronos RUL — FD002 — LOCAL / OFFLINE version

Runs entirely on your PC (CPU), no internet needed after the models are cached once.
- Models loaded from local HuggingFace cache (offline mode)
- Embeddings cached to disk (survive restarts)
- Results written to `results/offline/` only (safe to git-push without conflicts)

**Author**: Fatima Khadija Benzine

---
## Setup (LOCAL / OFFLINE)

In [ ]:
# === Setup: LOCAL / OFFLINE ===
import os

# We are already inside the cloned repo locally — no git clone, no Drive.
REPO = r'C:\Users\fatim\Desktop\RUL-Chronos'
os.chdir(os.path.join(REPO, 'src'))

# Force HuggingFace to use the local cache (no internet needed once models are cached)
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

from pathlib import Path
from datetime import datetime
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import json, time, warnings
warnings.filterwarnings('ignore')

import torch
from sklearn.linear_model import Ridge
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error
from xgboost import XGBRegressor
from sklearn.cluster import KMeans
from sklearn.preprocessing import MinMaxScaler

from data_loader import MultiDatasetLoader
from preprocessing import DataNormalizer

TIMESTAMP = datetime.now().strftime('%Y%m%d_%H%M')

# Local results dir (timestamped, git-ignored — for figures/scratch)
SAVE_DIR = os.path.join(REPO, 'results', f'Chronos_Eval_{TIMESTAMP}')
os.makedirs(SAVE_DIR, exist_ok=True)

# Fixed cache dir for embeddings (survives restarts, git-ignored)
EMB_ROOT = os.path.join(REPO, 'results', 'embeddings_cache', 'FD002')
os.makedirs(EMB_ROOT, exist_ok=True)

# Dedicated OFFLINE results dir (this IS git-tracked — your safe file)
OFFLINE_DIR = os.path.join(REPO, 'results', 'offline')
os.makedirs(OFFLINE_DIR, exist_ok=True)

DEVICE = 'cpu'   # local PC, CPU only

ALL_RESULTS = []  # collects every result row

plt.style.use('seaborn-v0_8-whitegrid')

print(f"Device: {DEVICE}")
print(f"Repo: {REPO}")
print(f"Offline results -> {OFFLINE_DIR}")
print("Setup complete (offline mode) ✓")

---
## Part A — Data Preparation (FD002)

In [ ]:
# === A1: Load C-MAPSS FD002 ===
DATASET = 'FD002'          # 6 operating conditions
W = 30
RUL_CAP = 125
N_REGIMES = 6              # FD002 has 6 operating regimes

loader = MultiDatasetLoader()
ds = loader.load_cmapss_dataset(DATASET)

train_raw = ds['train'].copy()
test_raw = ds['test'].copy()

# Apply RUL cap
train_raw['rul'] = train_raw['rul'].clip(upper=RUL_CAP)
if 'rul' in test_raw.columns:
    test_raw['rul'] = test_raw['rul'].clip(upper=RUL_CAP)

sensor_cols = [c for c in train_raw.columns if c.startswith('sensor_')]
setting_cols = [c for c in train_raw.columns if c.startswith('setting_')]

print(f"✓ Data loaded: {DATASET}")
print(f"  Train: {len(train_raw)} samples, {train_raw['unit'].nunique()} units")
print(f"  Test:  {len(test_raw)} samples, {test_raw['unit'].nunique()} units")
print(f"  Sensors: {len(sensor_cols)}, Settings: {len(setting_cols)}")

In [ ]:
# === A2a: Identify the 6 Operating Regimes ===
# Cluster the 3 operating settings into 6 groups. Fit on TRAIN only,
# then assign the same clusters to TEST (no leakage).

kmeans = KMeans(n_clusters=N_REGIMES, random_state=42, n_init=10)
train_raw['regime'] = kmeans.fit_predict(train_raw[setting_cols].values)
test_raw['regime']  = kmeans.predict(test_raw[setting_cols].values)

print(f"✓ Identified {N_REGIMES} operating regimes")
print("\nRegime distribution (train):")
print(train_raw['regime'].value_counts().sort_index().to_string())
print("\nRegime distribution (test):")
print(test_raw['regime'].value_counts().sort_index().to_string())

In [ ]:
# === A2b: Visualize the 6 Operating Regimes ===
# Confirm the operating settings separate cleanly into 6 clusters.

fig = plt.figure(figsize=(14, 4))
colors = plt.cm.tab10(np.linspace(0, 1, N_REGIMES))

# 3D-style view via three 2D projections of the settings
pairs = [(0, 1), (0, 2), (1, 2)]
for idx, (a, b) in enumerate(pairs):
    ax = fig.add_subplot(1, 3, idx + 1)
    for r in range(N_REGIMES):
        mask = train_raw['regime'] == r
        ax.scatter(train_raw.loc[mask, setting_cols[a]],
                   train_raw.loc[mask, setting_cols[b]],
                   s=6, alpha=0.4, color=colors[r], label=f'Regime {r}')
    ax.set_xlabel(setting_cols[a])
    ax.set_ylabel(setting_cols[b])
    ax.set_title(f'{setting_cols[a]} vs {setting_cols[b]}')
    if idx == 2:
        ax.legend(markerscale=2, fontsize=8, loc='center left',
                  bbox_to_anchor=(1.02, 0.5))

plt.suptitle('FD002 Operating Regimes (K-Means, k=6)', fontweight='bold')
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/fd002_regimes_{TIMESTAMP}.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ The 6 regimes should appear as clearly separated clusters.")

In [ ]:
# === A2c: Per-Regime (Condition-Specific) Normalization ===
# Fit one MinMax scaler PER regime on sensors, using TRAIN only,
# then apply to both train and test. This removes the regime effect
# so the normalized signal reflects degradation, not operating condition.

scalers = {}
train_norm = train_raw.copy()
test_norm  = test_raw.copy()

for r in range(N_REGIMES):
    scaler = MinMaxScaler()
    tr_mask = train_raw['regime'] == r
    scaler.fit(train_raw.loc[tr_mask, sensor_cols].values)
    scalers[r] = scaler

    train_norm.loc[tr_mask, sensor_cols] = scaler.transform(
        train_raw.loc[tr_mask, sensor_cols].values)

    te_mask = test_raw['regime'] == r
    if te_mask.any():
        test_norm.loc[te_mask, sensor_cols] = scaler.transform(
            test_raw.loc[te_mask, sensor_cols].values)

# After per-regime normalization, sensors are the feature pool.
# (Settings only encoded the regime, which is now normalized out.)
feature_cols = sensor_cols

print("✓ Per-regime MinMax normalization complete")
print(f"  Feature pool: {len(feature_cols)} sensors")

# Sanity check: a sensor should now look like a degradation trend,
# not a step function jumping between regimes.
u = sorted(train_norm['unit'].unique())[0]
ud = train_norm[train_norm['unit'] == u].sort_values('cycle')
plt.figure(figsize=(12, 3))
for s in sensor_cols[:4]:
    plt.plot(ud['cycle'], ud[s], label=s, alpha=0.8)
plt.xlabel('Cycle'); plt.ylabel('Normalized value')
plt.title(f'Normalized sensors for unit {u} (should show smooth degradation trends)')
plt.legend(fontsize=8); plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/fd002_normcheck_{TIMESTAMP}.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# === A3: Feature Selection ===

def variance_filter(df, feature_cols, threshold=0.001):
    """Remove low-variance features"""
    variances = df[feature_cols].var()
    selected = variances[variances > threshold].index.tolist()
    removed = [f for f in feature_cols if f not in selected]
    return selected, removed

def correlation_filter(df, feature_cols, threshold=0.95):
    """Remove highly correlated features"""
    corr_matrix = df[feature_cols].corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    to_drop = [col for col in upper.columns if any(upper[col] > threshold)]
    selected = [f for f in feature_cols if f not in to_drop]
    return selected, to_drop

def aficv_selection(df, feature_cols, target_col='rul', threshold=0.90):
    """AFICv: Accumulated Feature Importance with Cross-Validation"""
    X = df[feature_cols].values
    y = df[target_col].values

    model = XGBRegressor(n_estimators=100, max_depth=5, random_state=42, n_jobs=-1)
    model.fit(X, y)

    importances = model.feature_importances_
    indices = np.argsort(importances)[::-1]

    cumsum = np.cumsum(importances[indices])
    n_features = np.searchsorted(cumsum, threshold) + 1

    selected_indices = indices[:n_features]
    selected = [feature_cols[i] for i in selected_indices]

    return selected, cumsum[n_features-1]

# Strategy 1: Variance + Correlation filtering
print("=== Strategy 1: Variance + Correlation Filtering ===")
feats_var, removed_var = variance_filter(train_norm, feature_cols, threshold=0.001)
print(f"  Variance filter: {len(feature_cols)} → {len(feats_var)} (removed: {removed_var})")

feats_corr, removed_corr = correlation_filter(train_norm, feats_var, threshold=0.95)
print(f"  Correlation filter: {len(feats_var)} → {len(feats_corr)} (removed: {removed_corr})")

# Strategy 2: AFICv
print("\n=== Strategy 2: AFICv (90% importance coverage) ===")
feats_aficv, coverage = aficv_selection(train_norm, feature_cols, threshold=0.90)
print(f"  Selected: {len(feats_aficv)} features (coverage: {coverage:.1%})")
print(f"  Features: {feats_aficv}")

# Store feature sets
feature_sets = {
    'correlation': feats_corr,
    'aficv': feats_aficv
}

print("\n=== Summary ===")
for name, feats in feature_sets.items():
    print(f"  {name}: {len(feats)} features")

In [ ]:
# === A4: Create Sliding Windows ===

def create_sliding_windows(df, feature_cols, window_size=30):
    """Create sliding windows for each unit"""
    X_list, y_list, unit_list = [], [], []

    for unit in sorted(df['unit'].unique()):
        unit_data = df[df['unit'] == unit].sort_values('cycle')
        n_samples = len(unit_data)

        if n_samples < window_size:
            continue

        values = unit_data[feature_cols].values
        ruls = unit_data['rul'].values

        for start in range(n_samples - window_size + 1):
            end = start + window_size
            X_list.append(values[start:end])
            y_list.append(ruls[end - 1])
            unit_list.append(unit)

    return np.array(X_list), np.array(y_list), np.array(unit_list)

# Create windows for each feature set
data_dict = {}

for fs_name, features in feature_sets.items():
    X_train, y_train, units_train = create_sliding_windows(train_norm, features, W)
    X_test, y_test, units_test = create_sliding_windows(test_norm, features, W)

    data_dict[fs_name] = {
        'X_train': X_train, 'y_train': y_train, 'units_train': units_train,
        'X_test': X_test, 'y_test': y_test, 'units_test': units_test,
        'features': features
    }

    print(f"{fs_name}: X_train={X_train.shape}, X_test={X_test.shape}")

print("\n✓ Sliding windows created")

In [ ]:
# === A5: Evaluation Helpers ===

def nasa_score(y_true, y_pred):
    """NASA asymmetric scoring function (Saxena et al. 2008)"""
    d = y_pred - y_true
    score = np.where(d < 0, np.exp(-d/13) - 1, np.exp(d/10) - 1)
    return np.sum(score)

def evaluate_per_unit(y_true, y_pred, unit_labels):
    """Evaluate using last prediction per unit (standard C-MAPSS protocol)"""
    preds_last, true_last = [], []
    for u in sorted(set(unit_labels)):
        mask = unit_labels == u
        if mask.sum() > 0:
            preds_last.append(y_pred[mask][-1])
            true_last.append(y_true[mask][-1])

    preds_last = np.array(preds_last)
    true_last = np.array(true_last)

    rmse = np.sqrt(mean_squared_error(true_last, preds_last))
    score = nasa_score(true_last, preds_last)
    return rmse, score

ALL_RESULTS = []
print("Evaluation helpers ✓")

---
## Part B — Chronos-1 (offline, cached)

Chronos-1 runs on **CPU** and is slow, but with the disk cache you only pay once.
If the cache files exist, they load instantly.

In [ ]:
# === B1: Load Chronos-1 (offline, CPU) ===
from chronos import ChronosPipeline

CHRONOS1_MODEL = "amazon/chronos-t5-small"
print(f"Loading Chronos-1: {CHRONOS1_MODEL} (from local cache)...")
chronos1_pipeline = ChronosPipeline.from_pretrained(
    CHRONOS1_MODEL,
    device_map='cpu',        # Chronos-1 is most reliable on CPU
    dtype=torch.float32
)
print("Chronos-1 loaded ✓")

In [ ]:
# === B2: Chronos-1 Embedding Function ===
def extract_chronos1_embeddings(pipeline, X, batch_size=64):
    """Univariate: embed each feature separately, mean-pool over time, concat features."""
    all_embeddings = []
    for i in range(0, len(X), batch_size):
        batch = X[i:i+batch_size]          # (B, W, F)
        feat_embs = []
        for f in range(batch.shape[2]):
            series = batch[:, :, f]        # (B, W)
            ts = torch.tensor(series, dtype=torch.float32)  # CPU
            with torch.no_grad():
                emb = pipeline.embed(ts)[0]           # tuple -> tensor  (B, W, dim)
                pooled = emb.mean(dim=1).cpu().numpy() # mean over time -> (B, dim)
            feat_embs.append(pooled)
        all_embeddings.append(np.concatenate(feat_embs, axis=1))  # concat features
        if (i // batch_size) % 10 == 0:
            print(f"    {min(i+batch_size, len(X))}/{len(X)}")
    return np.vstack(all_embeddings)

print("Chronos-1 embedding function defined ✓")

In [ ]:
# === B3: Chronos-1 Evaluation (with disk cache) ===
EMB_DIR_C1 = os.path.join(EMB_ROOT, 'chronos1')
os.makedirs(EMB_DIR_C1, exist_ok=True)

def get_or_compute_c1(X, cache_path):
    if os.path.exists(cache_path):
        print(f"  ✓ cache: {os.path.basename(cache_path)}")
        return np.load(cache_path)
    print("  computing (will cache)...")
    emb = extract_chronos1_embeddings(chronos1_pipeline, X)
    np.save(cache_path, emb)
    return emb

for fs_name in feature_sets.keys():
    print(f"\n{'='*60}\nChronos-1 + {fs_name.upper()}\n{'='*60}")
    data = data_dict[fs_name]
    Xtr = get_or_compute_c1(data['X_train'], os.path.join(EMB_DIR_C1, f'train_{fs_name}.npy'))
    Xte = get_or_compute_c1(data['X_test'],  os.path.join(EMB_DIR_C1, f'test_{fs_name}.npy'))

    heads = [
        ('Ridge', Ridge(alpha=1.0)),
        ('MLP', MLPRegressor(hidden_layer_sizes=(256,128), max_iter=500, random_state=456)),
        ('XGBoost', XGBRegressor(n_estimators=100, max_depth=5, random_state=456))
    ]
    for hn, hm in heads:
        hm.fit(Xtr, data['y_train'])
        yp = hm.predict(Xte)
        rmse, score = evaluate_per_unit(data['y_test'], yp, data['units_test'])
        ALL_RESULTS.append({'Model':'Chronos-1','Feature_Set':fs_name,'Head':hn,
                            'Test_RMSE':round(rmse,2),'Test_Score':round(score,1)})
        print(f"  {hn}: RMSE={rmse:.2f}, Score={score:.1f}")

print("\n✓ Chronos-1 done")

---
## Part C — Chronos-2 (offline, cached)

In [ ]:
# === C1: Load Chronos-2 (offline, CPU) ===
from chronos import Chronos2Pipeline

CHRONOS2_MODEL = "amazon/chronos-2"
print(f"Loading Chronos-2: {CHRONOS2_MODEL} (from local cache)...")
chronos2_pipeline = Chronos2Pipeline.from_pretrained(
    CHRONOS2_MODEL,
    device_map='cpu'
)
print("Chronos-2 loaded ✓")

In [ ]:
# === C2: Chronos-2 Embedding Function (with correct list handling) ===
def extract_chronos2_embeddings(pipeline, X, batch_size=32):
    """Multivariate. embed() returns (list_of_tensors, loc_scale);
       each tensor is (n_features, n_patches, dim). Mean-pool over features+patches."""
    n_samples = X.shape[0]
    all_embeddings = []
    for i in range(0, n_samples, batch_size):
        batch = X[i:i+batch_size]
        batch_t = np.transpose(batch, (0, 2, 1))            # (B, F, W)
        ts_tensor = torch.tensor(batch_t, dtype=torch.float32)  # CPU (Chronos-2 handles device internally)
        with torch.no_grad():
            result = pipeline.embed(ts_tensor)
            emb_list = result[0]                    # list of (F, P, dim) tensors
            emb = torch.stack(emb_list)             # (B, F, P, dim)
            emb_pooled = emb.mean(dim=(1, 2)).cpu().numpy()  # -> (B, dim)  mean pooling
        all_embeddings.append(emb_pooled)
        if (i // batch_size) % 10 == 0:
            print(f"    {min(i+batch_size, n_samples)}/{n_samples}")
    return np.vstack(all_embeddings)

print("Chronos-2 embedding function defined ✓")

In [ ]:
# === C2b: Chronos-2 Pooling Strategies ===
def extract_chronos2_embeddings_v2(pipeline, X, batch_size=32, pooling='mean'):
    """pooling: 'mean' | 'last' | 'concat' | 'last_concat'"""
    n_samples = X.shape[0]
    all_embeddings = []
    for i in range(0, n_samples, batch_size):
        batch = X[i:i+batch_size]
        batch_t = np.transpose(batch, (0, 2, 1))
        ts_tensor = torch.tensor(batch_t, dtype=torch.float32)
        with torch.no_grad():
            result = pipeline.embed(ts_tensor)
            emb = torch.stack(result[0])            # (B, F, P, dim)
            if pooling == 'mean':
                out = emb.mean(dim=(1, 2))
            elif pooling == 'last':
                out = emb[:, :, -1, :].mean(dim=1)
            elif pooling == 'concat':
                out = emb.mean(dim=1).reshape(emb.shape[0], -1)
            elif pooling == 'last_concat':
                out = emb[:, :, -1, :].reshape(emb.shape[0], -1)
            out = out.cpu().numpy()
        all_embeddings.append(out)
    return np.vstack(all_embeddings)

print("Pooling strategies defined ✓")

In [ ]:
# === C3: Chronos-2 Evaluation (mean pooling, with disk cache) ===
EMB_DIR_C2 = os.path.join(EMB_ROOT, 'chronos2')
os.makedirs(EMB_DIR_C2, exist_ok=True)

def get_or_compute_c2(X, cache_path):
    if os.path.exists(cache_path):
        print(f"  ✓ cache: {os.path.basename(cache_path)}")
        return np.load(cache_path)
    print("  computing (will cache)...")
    emb = extract_chronos2_embeddings(chronos2_pipeline, X)
    np.save(cache_path, emb)
    return emb

all_predictions = {}

for fs_name in feature_sets.keys():
    print(f"\n{'='*60}\nChronos-2 + {fs_name.upper()}\n{'='*60}")
    all_predictions[fs_name] = {}
    data = data_dict[fs_name]

    Xtr = get_or_compute_c2(data['X_train'], os.path.join(EMB_DIR_C2, f'train_{fs_name}_mean.npy'))
    Xte = get_or_compute_c2(data['X_test'],  os.path.join(EMB_DIR_C2, f'test_{fs_name}_mean.npy'))

    heads = [
        ('Ridge', Ridge(alpha=1.0)),
        ('MLP', MLPRegressor(hidden_layer_sizes=(256,128), max_iter=500, random_state=456)),
        ('XGBoost', XGBRegressor(n_estimators=100, max_depth=5, random_state=456))
    ]
    for hn, hm in heads:
        hm.fit(Xtr, data['y_train'])
        yp = hm.predict(Xte)
        all_predictions[fs_name][hn] = {
            'y_pred': yp.copy(), 'y_test': data['y_test'].copy(),
            'units_test': data['units_test'].copy()}
        rmse, score = evaluate_per_unit(data['y_test'], yp, data['units_test'])
        ALL_RESULTS.append({'Model':'Chronos-2','Feature_Set':fs_name,'Head':hn,
                            'Test_RMSE':round(rmse,2),'Test_Score':round(score,1)})
        print(f"  {hn}: RMSE={rmse:.2f}, Score={score:.1f}")

print("\n✓ Chronos-2 done")

---
## Part D — Save Results to `results/offline/`

In [ ]:
# === Save all results to the dedicated OFFLINE file (git-safe) ===
df = pd.DataFrame(ALL_RESULTS)
print(df.to_string(index=False))

csv_path  = os.path.join(OFFLINE_DIR, 'fd002_results.csv')
json_path = os.path.join(OFFLINE_DIR, 'fd002_results.json')
df.to_csv(csv_path, index=False)
with open(json_path, 'w') as f:
    json.dump(ALL_RESULTS, f, indent=2)

print(f"\n✓ Saved to {csv_path}")
print(f"✓ Saved to {json_path}")
print("\nCommit only this folder:  git add results/offline/  &&  git commit  &&  git push")